In [1]:
import java.io.File

val file = File("sample_text")
val text = file.readText()

// was very useful
// https://docs.oracle.com/javase/8/docs/api/java/util/regex/Pattern.html

// also split by '\n' to avoid following
//      some sentence with number at the end 43.
//      41 other sentence with number at the start
// if '\n' split is not added, then we will have '43.\n41' instead of 2 separate entries
val rawSplitContent = text.split(" ", "\n").filter { it != "" }
// We don't want to bother with words like:
// 123123^^&1123123 (what?)
// 123.423.4321.54 (end of sentence, then floating point number, then again end of sentence?)
// 654,543,543  (might be just comma as sentence separation, or might be a floating point then comma?)

// in case potential splitted word ends with '.' or ','
// that means, it had whitspace characters around it.
// Therefore remove last trailing dot or comma to avoid
// skipping potential decimal. Example:
// "some text with kinda strange fractional at the end 43.51. Next sentence"
// here 43.51 should be normally processed
val splitContent = rawSplitContent.map { str ->
    str.trimEnd('.', ',')
}

// above text was split by whitespace characters.
// so use '^' and '$' (start and end of string) to check:
// 1. does potential number word have a sign?
// 2. does it start with digits?
// 3. does it have dot or comma?
// 4. if yes, do other digits follow it?
val regexp = """^[+-]?\d+([.,]\d+)?$""".toRegex()
// we may have text like: "some text with decimal at the end 43."
// then we should somehow understand that this last dot is full stop.
// Also, we might end up finding "some text with fractional at the end 43.21. Other text."
// but yet let us skip this moment
// it should be done at first iteration of parsing, rather than trying to make allmighty regular expression

// this function should be given result of regexp pass, i.e. used for internal logic
// after applying regular expression
// again, it must be better to separate responsibilities
// such that we don't try to write a function that can parse everything
// i.e. this function should only convert "normal" (or understandable) string
// representation of a decimal into actual Double type
// if something bad was given, then sorry, not the problem of this guy
//fun tryConvertToDobule(str: String) : Double? {
//
//}
fun convertToDouble(str: String) : Double {
    var sign = 1

    // don't want to check that in each
    // iteration of loop, in case str is actually a 'number'
    val noSignStr = if (str.startsWith('-')) {
        sign = -1
        str.substring(1)
    } else {
       str
    }

    val dotOrComma = noSignStr.indexOfFirst { ch -> ch == '.' || ch == ',' }

    val decimalStr = if (dotOrComma != -1) {
        noSignStr.substring(0, dotOrComma)
    } else {
        noSignStr
    }

    val fractionStr = if (dotOrComma != -1) {
        noSignStr.substring(dotOrComma + 1)
    } else {
        ""
    }

    val decimalPart = convertToDecimal(decimalStr)
    val fractionPart = convertToDecimal(fractionStr) / 10.0.pow(fractionStr.length)

    return sign * (decimalPart + fractionPart)
}

// helper function that assumes string representation
// of decimal (i.e. not fractional), to convert it into Double
fun convertToDecimal(str: String) : Double {
    var result : Double = 0.0

    for (ch in str) {
        result = result * 10 + (ch - '0')
    }

    return result
}

In [2]:
val numbersInText = mutableListOf<Double>()
val regexpMatches = splitContent.map { regexp.find(it) }.toList()

regexpMatches.forEach {
    it?.value?.let {
        numbersInText.add(convertToDouble(it))
    }
}

println(numbersInText)

[123.0, 12093.0, 43.32, 2093.0, 32.0, 14.0, 2.0, 5.0, 3.0, 2.0, 321.0, -123.5312, 4324.031, -4.121232300009E7, 584.3]
